# RG3 가스: 과거 구간 내부 센서 관계 변화

4쌍: 최대압력-속도 / 전환압력-최대압력 / 쿠션-계량위치 / 계량시간-계량위치. 각 history_segment 앞선 최대100샷(최소5)의 평균·분산·공분산으로 선형관계 기대값과 잔차를 계산. 현재 샷은 과거 통계에 미포함. 비라벨 포함 과거 센서만 사용하며 정상이라고 라벨링하지 않는다. 잔차 부호/절댓값8, std4, 전문가 최대 점수 앙상블. RF/ET 두 시드 평균.

RG3 RH 620개 가스19/정상601. CN7 미사용. history_segment는 센서 연속관측 분할, 평가구간은 라벨 관측 간격 분할. 온라인 센서 과거를 이용하는 평가로 평가 구간 이전 센서도 배포 시 관측된다는 가정. 평가 라벨은 어떤 피처 생성에도 미사용. 중첩 구간검증이지 새 독립 검증 아님. 결측은 학습 정상으로 대체하며 초반 기록부족 자체가 신호가 되는 것을 보장하지 않음.


In [1]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import ExtraTreesRegressor,RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/67_rg3_recall_first_nested_mode_ensemble.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.split('F=pd.DataFrame')[0])
assert len(d)==620 and y.sum()==19
inputs=['Max_Injection_Speed','Average_Screw_RPM','Barrel_Temperature_2','Barrel_Temperature_5','Plasticizing_Position','Filling_Time']
targets=['Max_Injection_Pressure','Max_Switch_Over_Pressure','Cushion_Position','Plasticizing_Time']
stdcols=[x+'__past10_std' for x in ['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']]
# Available historical fields: use metering position instead of unavailable switchover position/RPM.
pairs=[('Max_Injection_Pressure','Max_Injection_Speed'),('Max_Switch_Over_Pressure','Max_Injection_Pressure'),('Cushion_Position','Plasticizing_Position'),('Plasticizing_Time','Plasticizing_Position')]
localcols=[]
for target,predictor in pairs:
 x=observed[predictor];z=observed[target];seg=observed.history_segment
 def past(a):
  prior=a.groupby(seg).shift(1)
  return prior.groupby(seg).transform(lambda v:v.rolling(100,min_periods=5).mean())
 mx=past(x);mz=past(z);vx=(past(x*x)-mx*mx).clip(lower=0);vz=(past(z*z)-mz*mz).clip(lower=0);cov=past(x*z)-mx*mz
 beta=cov/(vx+1e-6)
 expected=mz+beta*(x-mx)
 noise=np.sqrt((vz-cov*cov/(vx+1e-6)).clip(lower=0))
 denom=np.maximum(noise,np.maximum(np.sqrt(vz)*.3,1e-6))
 name=target+'__local_relation_residual'
 observed[name]=(z-expected)/denom
 localcols.append(name)
F=d[['EQUIP_CD','product','TimeStamp']].merge(observed[['EQUIP_CD','product','TimeStamp']+localcols],on=['EQUIP_CD','product','TimeStamp'],validate='one_to_one')[localcols].replace([np.inf,-np.inf],np.nan)
F.assign(TimeStamp=d.TimeStamp).to_csv(tables/'rg3_causal_local_relation_features.csv',index=False,encoding='utf-8-sig')
memo={};audit=[]
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 normal=ref[y[ref]==0]
 im=SimpleImputer(strategy='median',keep_empty_features=True).fit(F.iloc[normal])
 rt=np.clip(im.transform(F.iloc[ref]),-20,20);rq=np.clip(im.transform(F.iloc[q]),-20,20)
 si=SimpleImputer(strategy='median',keep_empty_features=True).fit(d.iloc[normal][stdcols])
 st=si.transform(d.iloc[ref][stdcols]);sq=si.transform(d.iloc[q][stdcols])
 features=[(st,sq),(np.c_[rt,np.abs(rt)],np.c_[rq,np.abs(rq)])]
 scores=[]
 for ft,fq in features:
  vals=[]
  for cls in [RandomForestClassifier,ExtraTreesClassifier]:
   for seed in [42,7]:
    if len(np.unique(y[ref]))<2:vals.append(np.zeros(len(q)));continue
    m=cls(n_estimators=80,max_depth=4,min_samples_leaf=4,class_weight='balanced',random_state=seed,n_jobs=2)
    vals.append(m.fit(ft,y[ref]).predict_proba(fq)[:,1])
  scores.append(np.mean(vals,axis=0))
 memo[key]=np.column_stack([scores[0],scores[1],np.maximum(*scores)])
 audit.append(dict(train=len(ref),query=len(q),normal_imputer_training=len(normal),gas_training=int(y[ref].sum()),train_blocks=','.join(map(str,groups.iloc[ref].unique())),query_blocks=','.join(map(str,groups.iloc[q].unique()))))
 return memo[key]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b))
 out=predict(ref,q);inner=np.full((len(ref),3),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy();inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for k,name in enumerate(['std4_reference','causal_local_relation8','std4_local_relation_max_ensemble']):
  for budget in [.02,.05,.1]:
   cut=float(np.quantile(inner[y[ref]==0,k],1-budget,method='higher'))
   pp=inner[:,k]>cut;tn,fp,fn,tp=confusion_matrix(y[ref],pp,labels=[0,1]).ravel()
   rec=tp/(tp+fn) if tp+fn else 0;f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
   candidates.append((rec,f1,-fp,name,budget,out[:,k]>cut))
   for j,i in enumerate(q):records.append(dict(model=name,budget=budget,TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(out[j,k]>cut),score=float(out[j,k]),threshold=cut))
 chosen=max(candidates,key=lambda x:x[:3])
 choices.append(dict(block=int(b),selected=chosen[3],budget=chosen[4],inner_recall=chosen[0],inner_F1=chosen[1]))
 for j,i in enumerate(q):records.append(dict(model='inner_recall_first_selected',budget=0,TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(chosen[5][j]),score=np.nan,threshold=np.nan))
 print('Finished RG3 conditional relationship block',b,flush=True)
p=pd.DataFrame(records);rows=[]
for (name,budget),g in p.groupby(['model','budget']):
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,budget=budget,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),recall=tp/(tp+fn),F1=2*tp/(2*tp+fp+fn),AP=average_precision_score(g.actual,g.score) if g.score.notna().all() else np.nan))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(audit),'training_audit')]:obj.to_csv(tables/f'rg3_causal_local_relation_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))



Finished RG3 conditional relationship block 1


Finished RG3 conditional relationship block 2


Finished RG3 conditional relationship block 3


Finished RG3 conditional relationship block 4


Finished RG3 conditional relationship block 5


Finished RG3 conditional relationship block 6


Finished RG3 conditional relationship block 7


                           model  budget  TP  FP  FN  TN   recall       F1       AP
          causal_local_relation8    0.02   0   9  19 592 0.000000 0.000000 0.029961
          causal_local_relation8    0.05   0  23  19 578 0.000000 0.000000 0.029961
          causal_local_relation8    0.10   2  55  17 546 0.105263 0.052632 0.029961
     inner_recall_first_selected    0.00   3  63  16 538 0.157895 0.070588      NaN
std4_local_relation_max_ensemble    0.02   3  26  16 575 0.157895 0.125000 0.060535
std4_local_relation_max_ensemble    0.05   3  33  16 568 0.157895 0.109091 0.060535
std4_local_relation_max_ensemble    0.10   3  65  16 536 0.157895 0.068966 0.060535
                  std4_reference    0.02   3  26  16 575 0.157895 0.125000 0.061578
                  std4_reference    0.05   3  33  16 568 0.157895 0.109091 0.061578
                  std4_reference    0.10   3  67  16 534 0.157895 0.067416 0.061578
